# Pengujian Forward Propagation RNN from Scratch 

#### Import Dataset

In [1]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, SimpleRNN, Dropout, Dense
from tensorflow.keras import Sequential, Input
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# impor modul dari folder src/model
import sys
import os
sys.path.append(os.path.abspath("../../src"))
from model.Model import Model

Seeding

In [2]:
import random
import os

# Atur seed
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.keras.utils.set_random_seed(seed_value)
tf.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '42'

#### Load Dataset

In [3]:
# Pemetaan label ke angka
label_mapping = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

# Load Dataset NusaX-Sentiment
# Train dataset
df_train = pd.read_csv("../../datasets/indonesian/train.csv")
train_texts = df_train['text'].astype(str).tolist()
df_train['label'] = df_train['label'].map(label_mapping) # Pemetaan label ke angka
train_labels = df_train['label'].astype(int).tolist()

# Validation dataset
df_val = pd.read_csv("../../datasets/indonesian/valid.csv")
val_texts = df_val['text'].astype(str).tolist()
df_val['label'] = df_val['label'].map(label_mapping) # Pemetaan label ke angka
val_labels = df_val['label'].astype(int).tolist()

# Test dataset
df_test = pd.read_csv("../../datasets/indonesian/test.csv")
test_texts = df_test['text'].astype(str).tolist()
df_test['label'] = df_test['label'].map(label_mapping) # Pemetaan label ke angka
test_labels = df_test['label'].astype(int).tolist()

#### Preprocessing - Tokenization & Vectorization

In [4]:
# Split data
X_train, X_val, X_test, y_train, y_val, y_test = train_texts, val_texts, test_texts, train_labels, val_labels, test_labels

# TextVectorization (Tokenization)
MAX_TOKENS = 10000      # Jumlah maksimum kata unik
MAX_LEN = 100           # Panjang maksimum sequence

vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    pad_to_max_tokens=True,
    ngrams=None,
)
vectorizer.adapt(X_train)

In [5]:
# 4. Dataset pipeline
def prepare_dataset(X, y):
    X_vec = vectorizer(tf.constant(X))
    return tf.data.Dataset.from_tensor_slices((X_vec, y)).batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = prepare_dataset(X_train, y_train)
val_ds = prepare_dataset(X_val, y_val)
test_ds = prepare_dataset(X_test, y_test)

In [6]:
EMBED_DIM = 128

## Test 1

### Model Keras

In [7]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_TOKENS, output_dim=EMBED_DIM),  # Embedding layer
    Bidirectional(SimpleRNN(64, return_sequences=True)),    # Bidirectional SimpleRNN layer
    Bidirectional(SimpleRNN(32)),
    Dropout(0.5),                                           # Dropout layer
    Dense(3, activation='softmax')                          # Dense layer
])


model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    batch_size=32
)

# Prediksi di test set
predictions = model.predict(test_ds)
predicted_labels = np.argmax(predictions, axis=1)

# Hitung macro F1-Score
from sklearn.metrics import f1_score
macro_f1 = f1_score(test_labels, predicted_labels, average='macro')
print(f"Macro F1-Score: {macro_f1:.4f}")

# save model
model.save('RNN_model1.h5')

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.3893 - loss: 1.1516 - val_accuracy: 0.5100 - val_loss: 0.9527
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6505 - loss: 0.8367 - val_accuracy: 0.5100 - val_loss: 0.9975
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8795 - loss: 0.4699 - val_accuracy: 0.5500 - val_loss: 1.0077
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9774 - loss: 0.2009 - val_accuracy: 0.5400 - val_loss: 1.0367
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9973 - loss: 0.0856 - val_accuracy: 0.5400 - val_loss: 1.1288
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 1.0000 - loss: 0.0430 - val_accuracy: 0.5400 - val_loss: 1.2092
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9987 - loss: 0.0296 - val_accuracy: 0.5600 - val_loss: 1.2247
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 1.0000 - loss: 0.0264 - val_accuracy: 0.5600 - v

Macro F1-Score: 0.4456


### Model Scratch

In [8]:
my_model = Model()
my_model.set_input_shape((MAX_LEN,))
my_model.load('RNN_model1.h5')

# Prediksi di test set
X_test_vector = vectorizer(X_test)
y_pred_probs = my_model.predict(X_test_vector.numpy())
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(test_labels, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

Processing layer 1: Embedding
Processing layer 2: Bidirectional
data processed: 100/100 timesteps
data processed: 100/100 timesteps
Processing layer 3: Bidirectional
data processed: 100/100 timesteps
data processed: 100/100 timesteps
Processing layer 4: Dense
Macro F1-Score di Test Set: 0.4456


## Test 2

### Model Keras

In [9]:
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_TOKENS, output_dim=EMBED_DIM),  # Embedding layer
    SimpleRNN(64, return_sequences=True),                   # Bidirectional SimpleRNN layer
    SimpleRNN(32),
    Dropout(0.5),                                           # Dropout layer
    Dense(3, activation='softmax')                          # Dense layer
])


model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    batch_size=32
)

# Prediksi di test set
predictions = model.predict(test_ds)
predicted_labels = np.argmax(predictions, axis=1)

# Hitung macro F1-Score
from sklearn.metrics import f1_score
macro_f1 = f1_score(test_labels, predicted_labels, average='macro')
print(f"Macro F1-Score: {macro_f1:.4f}")

# save model
model.save('RNN_model2.h5')

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.3607 - loss: 1.1994 - val_accuracy: 0.4300 - val_loss: 1.0934
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.4832 - loss: 1.0127 - val_accuracy: 0.3100 - val_loss: 1.2500
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7190 - loss: 0.7143 - val_accuracy: 0.3600 - val_loss: 1.3305
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.7164 - loss: 0.6964 - val_accuracy: 0.3300 - val_loss: 1.3469
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.6486 - loss: 0.7693 - val_accuracy: 0.3300 - val_loss: 1.5392
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6178 - loss: 0.7751 - val_accuracy: 0.3100 - val_loss: 1.7219
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6034 - loss: 0.8506 - val_accuracy: 0.4200 - val_loss: 1.5563
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5598 - loss: 0.9591 - val_accuracy: 0.4000 - v

Macro F1-Score: 0.3217


### Model Scratch

In [10]:
my_model = Model()
my_model.set_input_shape((MAX_LEN,))
my_model.load('RNN_model2.h5')

# Prediksi di test set
X_test_vector = vectorizer(X_test)
y_pred_probs = my_model.predict(X_test_vector.numpy())
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(test_labels, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

Processing layer 1: Embedding
Processing layer 2: SimpleRNN
data processed: 100/100 timesteps
Processing layer 3: SimpleRNN
data processed: 100/100 timesteps
Processing layer 4: Dense
Macro F1-Score di Test Set: 0.3217
